# Arrhythmia: raw Top-1 and Top-3 disagreement

This notebook reproduces the approved **Appendix-only** two-panel figure from the saved Arrhythmia SHAP explanations. It uses MLP/FTT, Random/CTE, five background seeds, $K=100$, $N=84$ queries and $d=278$ features. The documented KernelSHAP setup requests 1,024 coalitions and `l1_reg="num_features(10)"`.

Each panel reports a raw event rate over all $\binom{5}{2}N=840$ run-pair–instance comparisons. Top-1 counts different maximum-absolute-SHAP features; Top-3 counts different sets of the three largest absolute SHAP values. These are **not calibrated baseline exceedance rates**. No Mallows/nominal-d calibration, model fitting, SHAP generation, Monte Carlo or source-data modification occurs.

Run all cells from this notebook directory or anywhere inside the repository checkout. The original notebook and manuscript tables are preserved.

## 1. Locate the checkout and import the analysis helper

Only an ancestor containing `.shap_multiplicity_root`, `results_tree`, and the new helper is accepted. All outputs are confined to `appendix_e/`; the research environment is not modified.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "appendix-e-mpl"))

def locate_checkout():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if ((candidate / ".shap_multiplicity_root").is_file()
            and (candidate / "results_tree").is_dir()
            and (candidate / "analysis/appendix_e/arrhythmia.py").is_file()):
            return candidate
    raise FileNotFoundError("Open this notebook within the repository checkout.")

PROJECT_ROOT = locate_checkout()
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))
import matplotlib
matplotlib.use("Agg")  # Headless export; the saved PNG is displayed below.
import numpy as np
import pandas as pd
from IPython.display import display, Image
from appendix_e.common import FIGURE_DIR
from appendix_e.arrhythmia import (
    load_saved_runs, compute_metrics, validate_existing_summary,
    export_results, plot_churn,
)
print("Project:", PROJECT_ROOT)
print("NumPy:", np.__version__)
print("Analysis: saved SHAP only; no calibration or new explanations")

## 2. Require complete, aligned inputs

All 20 canonical `results_tree/arrhythmia_{model}_0_0_{method}_s{seed}_sv.pkl` files must exist. The loader requires every array and stored raw query matrix to have shape `(84, 278)`, all values to be finite, exact equality of raw query values/order, and identical feature names/order. It does not trim to a minimum row count or silently omit seeds.

Support is checked both as exact nonzero values and as `abs(SHAP) > 1e-10`, matching the historical notebook. Both checks must find exactly ten features in each of the 1,680 seed–query explanations. File SHA-256 hashes are recorded and checked again before export.

The historical Top-1 convention is `np.argmax(abs(SHAP))`; historical Top-3 Jaccard uses the last three indices of the default ascending `np.argsort(abs(SHAP))`. Selection/rank churn uses the first three of `np.argsort(-abs(SHAP))`. Both conventions are retained. The loader verifies that no Top-1 maximum or Top-3 boundary is tied and that the two Top-3 selections agree on these inputs; the 268 zero values are never selected into Top-3.

In [ ]:
runs, sources, input_checks = load_saved_runs(PROJECT_ROOT)
display(input_checks)
assert len(sources) == 20
print("Validated: 4 groups × 5 seeds × 84 queries × 278 features")
print("All raw query and feature positions align; every support has size 10.")

## 3. Compute exact counts and discrete Jaccard histograms

For Top-3 sets of size three, intersection sizes $3,2,1,0$ correspond to Jaccard distances $0,0.5,0.8,1.0$. A set-disagreement event is exactly `distance > 0`. Integer counts are exported for every category and for the total disagreement rate.

The historical `compute_jaccard_distance` function is extracted from `plot_violin.py` using its syntax tree and executed in an isolated NumPy namespace; the rest of that module is not run. Every one of the 3,360 pair–query distances must match it exactly.

In [ ]:
results = compute_metrics(PROJECT_ROOT, runs)
counts = results["counts"]
display(counts[["model", "method", "top1_pair_numerator", "top3_disagreement_numerator",
                "pair_denominator", "top1_pair_percent", "top3_disagreement_percent"]])
display(results["jaccard_histogram"])
assert len(results["pair_observations"]) == 3360
assert (results["jaccard_histogram"].groupby(["model", "method"]).numerator.sum() == 840).all()

## 4. Preserve and reproduce the support and conditional summaries

Top-1 any-seed uses $N=84$, while pairwise Top-1 and support similarity use $10N=840$. Mean support Jaccard similarity is the arithmetic mean of the individual support-intersection/support-union ratios, not a threshold exceedance rate.

Selection and rank churn are conditional on **Top-3-disagreeing** pair–query observations. A selection event contains a differing Top-3 feature absent from the other support; a rank event contains a differing Top-3 feature present in both supports. Events may belong to both categories. Their fractions are not additive components of a 100% decomposition.

All 20 values in `figures_b/arrhythmia_analysis/summary.json` must match within `5e-15`; exact new numerators and their distinct denominators are retained.

In [ ]:
historical_comparison = validate_existing_summary(PROJECT_ROOT, counts)
display(counts[["model", "method", "top1_any_numerator", "top1_any_denominator",
                "support_mean_jaccard_similarity", "selection_numerator", "rank_numerator",
                "selection_and_rank_numerator", "selection_rank_denominator",
                "selection_churn_fraction", "rank_churn_fraction"]])
print("Existing summary values reproduced:", len(historical_comparison))
print("Maximum absolute difference:", historical_comparison.absolute_difference.max())

## 5. Export the data and validation record

The event-count table, raw pair observations (query positions only, without raw feature values), Top-3 category histogram, support-intersection histogram, input checks, and comparison with the existing summary are saved as CSV. JSON files record the complete summary, source paths and SHA-256 values, software versions, historical tie conventions and validation results.

The support histogram preserves the integer numerator and denominator behind every possible support Jaccard ratio. The pair observation file preserves the overlap between conditional selection and rank events.

In [ ]:
output_dir, validation = export_results(
    PROJECT_ROOT, results, sources, input_checks, historical_comparison
)
display(validation)
print("Data directory:", output_dir)
assert validation["status"] == "passed"
assert validation["source_hashes_still_match"]
assert not validation["calibration_applied"]

## 6. Draw the two-panel appendix figure

Rows are MLP/FTT and columns are Random/CTE. Both panels share the fixed 0–100% `YlOrRd` scale and use the same denominator, 840. Cell labels give one decimal place; exported CSV/JSON files retain exact integer counts and full-precision rates. The support and conditional summaries remain separate from this figure.

In [ ]:
import matplotlib.pyplot as plt
figure = plot_churn(PROJECT_ROOT, counts)
plt.close(figure)
display(Image(filename=str(FIGURE_DIR / "appendix_arrhythmia_churn.png")))
print("Saved vector figure:", FIGURE_DIR / "appendix_arrhythmia_churn.pdf")
print("Saved review image:", FIGURE_DIR / "appendix_arrhythmia_churn.png")

## Interpretation and limits

Pairwise Top-1 changes occur in 34.0–37.9% of comparisons, and Top-3 set changes in 78.3–83.9%. All explanations contain ten nonzero features, but the selected supports have mean Jaccard similarity 0.335–0.362. These results describe one Arrhythmia test fold and the documented sparse explanation setting; they do not isolate a causal effect of dimensionality or establish a full-ranking calibrated result.

Original inputs, the original analysis notebook, existing summaries and manuscript table values remain unchanged. The large number of tied zero attributions is why no nominal-d Mallows calibration is introduced here.